# 🏨 Modelos Supervisados con Datos de Hotel
## Clasificación de cancelaciones · Regresión de precios · Comparación de algoritmos

---

En esta actividad trabajarás con el mismo dataset de **500 reservas de hotel** que en la actividad anterior y aplicarás técnicas de aprendizaje **supervisado**: esta vez el modelo aprende a partir de ejemplos etiquetados.

| Técnica | Algoritmo | ¿Para qué? |
|---|---|---|
| 🔵 Clasificación | Regresión Logística | Predecir si una reserva se cancelará |
| 🟢 Clasificación | Árbol de Decisión | Idem, con reglas interpretables |
| 🟠 Clasificación | Random Forest | Idem, combinando múltiples árboles |
| 🔴 Regresión | Regresión Lineal / Árbol / Random Forest | Predecir el precio por noche |

> **En el aprendizaje supervisado SÍ hay respuesta correcta.** El modelo aprende de ejemplos pasados (features + etiqueta) y generaliza a nuevos casos.

**⏱ Duración:** 90 minutos | **🐍 Lenguaje:** Python | **☁️ Entorno:** Google Colab

> Ejecuta cada celda en orden con `Shift + Enter`. Lee los bloques de texto antes de ejecutar.

---
## ⚙️ Instalación de dependencias
Colab ya incluye scikit-learn, pandas y matplotlib. Forzamos versiones estables.

In [ ]:
!pip install -q scikit-learn pandas numpy matplotlib seaborn
print('✅ Librerías listas')

---
## PARTE 1 · Generar el dataset de hotel 🏨
**⏱ 5 minutos**

Usamos el mismo generador que en la actividad anterior (misma semilla, mismas columnas) pero añadimos dos columnas nuevas que serán nuestros **targets**:

- `cancelado` → variable booleana para clasificación (¿se canceló la reserva?)
- `precio_noche` → variable numérica ya existente, usada para regresión

La columna `cancelado` se genera con probabilidad basada en factores reales de negocio hotelero:
reservas de última hora, canal OTA, sin historial previo y precio elevado aumentan la probabilidad.

In [ ]:
import pandas as pd
import numpy as np
import random
import warnings
warnings.filterwarnings('ignore')

# Semillas fijas para reproducibilidad
random.seed(42)
np.random.seed(42)

tipos_hotel      = ['Resort', 'City Hotel']
meses            = ['Enero','Febrero','Marzo','Abril','Mayo','Junio',
                    'Julio','Agosto','Septiembre','Octubre','Noviembre','Diciembre']
paises           = ['España','Francia','Portugal','Alemania','Italia','Reino Unido','USA','Brasil']
canales          = ['Directo','OTA','Agencia','Corporativo']
tipos_habitacion = ['Individual','Doble','Suite','Familiar']

rows = []
for i in range(500):
    tipo         = random.choice(tipos_hotel)
    mes          = random.choice(meses)
    noches       = random.randint(1, 14)
    adultos      = random.randint(1, 4)
    ninos        = random.randint(0, 2)
    pais         = random.choice(paises)
    canal        = random.choice(canales)
    habitacion   = random.choice(tipos_habitacion)
    precio_noche = round(random.uniform(50, 400), 2)
    precio_total = round(precio_noche * noches, 2)
    solicitudes  = random.randint(0, 5)
    prev_reservas= random.randint(0, 10)
    anticipacion = random.randint(0, 365)
    estrellas    = random.choice([3, 4, 5])
    valoracion   = round(random.uniform(5.0, 10.0), 1)

    rows.append([i+1, tipo, mes, noches, adultos, ninos, pais, canal,
                 habitacion, precio_noche, precio_total,
                 solicitudes, prev_reservas, anticipacion,
                 estrellas, valoracion])

columnas = ['id','tipo_hotel','mes_llegada','noches','adultos','ninos',
            'pais_origen','canal_reserva','tipo_habitacion','precio_noche',
            'precio_total','solicitudes_especiales','reservas_previas',
            'dias_anticipacion','estrellas_hotel','valoracion_cliente']

df = pd.DataFrame(rows, columns=columnas)
print(f'✅ Dataset base generado: {df.shape[0]} filas · {df.shape[1]} columnas')

In [ ]:
# Generar el target de clasificación: probabilidad de cancelación
# Basado en 4 factores de riesgo reales en la industria hotelera
prob_cancelacion = (
    (df['dias_anticipacion'] < 30).astype(float) * 0.30 +   # Reserva de última hora
    (df['canal_reserva'] == 'OTA').astype(float) * 0.20 +   # Canal con más cancelaciones
    (df['reservas_previas'] == 0).astype(float) * 0.20 +    # Sin historial = menos compromiso
    (df['precio_total'] > 500).astype(float) * 0.10         # Precio alto = más arrepentimiento
)
# Limitar probabilidad entre 0 y 1
prob_cancelacion = prob_cancelacion.clip(0, 1)

# Asignar cancelación con esa probabilidad (experimento de Bernoulli)
df['cancelado'] = np.random.binomial(1, prob_cancelacion).astype(bool)

n_canceladas = df['cancelado'].sum()
print(f'✅ Target de clasificación creado')
print(f'   Reservas canceladas : {n_canceladas} ({n_canceladas/len(df)*100:.1f}%)')
print(f'   Reservas activas    : {len(df)-n_canceladas} ({(len(df)-n_canceladas)/len(df)*100:.1f}%)')
print(f'\nPrimeras 5 filas:')
df[['id','canal_reserva','dias_anticipacion','reservas_previas','precio_total','cancelado']].head()

### 💬 Reflexión 1
> **¿Cuál de los 4 factores de riesgo crees que influye más en las cancelaciones reales de hoteles?**
> ¿Tiene sentido que un canal OTA tenga más cancelaciones que una reserva directa?
>
> *(Escribe tu respuesta aquí haciendo doble clic en esta celda)*

---

---
## PARTE 2 · Preprocesamiento: encoding, split y escalado 🔧
**⏱ 10 minutos**

Los algoritmos de ML solo entienden números. Antes de entrenar debemos:

1. **Codificar variables categóricas** con `pd.get_dummies()` (One-Hot Encoding)
2. **Separar features y targets** — un conjunto para clasificación, otro para regresión
3. **Dividir en train/test** (80% entrenamiento, 20% evaluación)
4. **Normalizar con StandardScaler** — necesario para Regresión Logística y Regresión Lineal

### ¿Por qué One-Hot Encoding?
Si codificamos `['Resort'=1, 'City Hotel'=2]`, el modelo interpreta que City Hotel "vale el doble" que Resort. One-Hot evita ese sesgo creando una columna binaria por categoría.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Columnas categóricas a codificar
cols_cat = ['tipo_hotel','mes_llegada','pais_origen','canal_reserva','tipo_habitacion']

# One-Hot Encoding: crea columnas binarias por cada categoría
df_encoded = pd.get_dummies(df, columns=cols_cat, drop_first=True)

print(f'Columnas antes del encoding : {df.shape[1]}')
print(f'Columnas después del encoding: {df_encoded.shape[1]}')
print('✅ Variables categóricas codificadas')

In [ ]:
# === PREPARACIÓN PARA CLASIFICACIÓN ===
# Target: cancelado (bool)
# Features: todo excepto id, cancelado, precio_noche y precio_total
# (precio_total se excluye porque es proporcional a precio_noche*noches → fuga de info)

TARGET_CLAS = 'cancelado'
excluir_clas = ['id', 'cancelado', 'precio_noche', 'precio_total']

X_clas = df_encoded.drop(columns=excluir_clas)
y_clas = df_encoded[TARGET_CLAS].astype(int)   # True→1, False→0

# División 80/20 con stratify para mantener proporción de cancelaciones
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_clas, y_clas,
    test_size=0.20,       # 20% para evaluación
    random_state=42,      # Reproducibilidad
    stratify=y_clas       # Igual % cancelaciones en train y test
)

# Normalización (necesaria para Regresión Logística)
scaler_c = StandardScaler()
X_train_c_sc = scaler_c.fit_transform(X_train_c)   # fit+transform en train
X_test_c_sc  = scaler_c.transform(X_test_c)         # solo transform en test

print('=== CONJUNTO DE CLASIFICACIÓN ===')
print(f'Features              : {X_clas.shape[1]}')
print(f'Train                 : {X_train_c.shape[0]} reservas')
print(f'Test                  : {X_test_c.shape[0]} reservas')
print(f'Canceladas en train   : {y_train_c.sum()} ({y_train_c.mean()*100:.1f}%)')
print(f'Canceladas en test    : {y_test_c.sum()} ({y_test_c.mean()*100:.1f}%)')
print('✅ Split de clasificación listo')

In [ ]:
# === PREPARACIÓN PARA REGRESIÓN ===
# Target: precio_noche (continuo)
# Features: todo excepto id, precio_noche, precio_total y cancelado

TARGET_REG = 'precio_noche'
excluir_reg = ['id', 'precio_noche', 'precio_total', 'cancelado']

X_reg = df_encoded.drop(columns=excluir_reg)
y_reg = df_encoded[TARGET_REG]

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg,
    test_size=0.20,
    random_state=42
)

# Normalización para regresión lineal
scaler_r = StandardScaler()
X_train_r_sc = scaler_r.fit_transform(X_train_r)
X_test_r_sc  = scaler_r.transform(X_test_r)

print('=== CONJUNTO DE REGRESIÓN ===')
print(f'Features            : {X_reg.shape[1]}')
print(f'Train               : {X_train_r.shape[0]} reservas')
print(f'Test                : {X_test_r.shape[0]} reservas')
print(f'Precio promedio     : €{y_reg.mean():.2f}/noche')
print(f'Rango de precios    : €{y_reg.min():.2f} – €{y_reg.max():.2f}')
print('✅ Split de regresión listo')

### 💬 Reflexión 2
> 1. **¿Por qué NO incluimos `precio_total` como feature para predecir `precio_noche`?** Pista: `precio_total = precio_noche × noches`.
> 2. **¿Por qué usamos `stratify=y_clas` en el split de clasificación pero no en el de regresión?**
>
> *(Escribe tu respuesta aquí)*

---

---
## PARTE 3 · Regresión Logística: clasificar cancelaciones 🔵
**⏱ 8 minutos**

Pese a su nombre, la **Regresión Logística** es un clasificador. Calcula la probabilidad de que una reserva sea cancelada usando la función sigmoide, que convierte cualquier número real en un valor entre 0 y 1.

```
Features → combinación lineal → sigmoide → probabilidad → umbral 0.5 → clase (0 o 1)
```

**Ventaja:** muy interpretable, los coeficientes indican la dirección del efecto de cada feature.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

# Entrenar Regresión Logística
# max_iter=1000 para asegurar convergencia con muchas features
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_c_sc, y_train_c)   # Entrenamiento con datos normalizados

# Predecir en el conjunto de test
y_pred_lr = lr.predict(X_test_c_sc)

print('=== REGRESIÓN LOGÍSTICA ===')
print(classification_report(y_test_c, y_pred_lr,
                             target_names=['No cancelada (0)','Cancelada (1)']))
print('✅ Regresión Logística entrenada')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Matriz de confusión ---
cm_lr = confusion_matrix(y_test_c, y_pred_lr)
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Pred: No cancel.','Pred: Cancelada'],
            yticklabels=['Real: No cancel.','Real: Cancelada'])
axes[0].set_title('Matriz de Confusión\nRegresión Logística', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Valor real')
axes[0].set_xlabel('Predicción')

# --- Top 15 coeficientes más importantes ---
coefs = pd.Series(lr.coef_[0], index=X_clas.columns)
top15 = coefs.abs().nlargest(15).index
coefs_top = coefs[top15].sort_values()

colores = ['#D32F2F' if v > 0 else '#1565C0' for v in coefs_top]
axes[1].barh(coefs_top.index, coefs_top.values, color=colores, edgecolor='white')
axes[1].axvline(x=0, color='black', linewidth=0.8)
axes[1].set_title('Top 15 coeficientes\n(rojo = aumenta cancelación)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Coeficiente')
axes[1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

---
## PARTE 4 · Árbol de Decisión: reglas interpretables 🟢
**⏱ 10 minutos**

Un **Árbol de Decisión** aprende reglas del tipo *"si canal=OTA Y anticipación < 30 → predice cancelado"*. Es el algoritmo más interpretable de todos: puedes ver exactamente qué decisiones toma.

**Parámetro clave:** `max_depth` controla la profundidad del árbol. Un árbol muy profundo memoriza los datos de entrenamiento (overfitting); uno muy superficial no captura patrones (underfitting).

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score

# Comparar distintas profundidades
profundidades = [2, 3, 4, 5, 8, None]   # None = sin límite
print('=== EFECTO DE max_depth EN EL ÁRBOL ===')
print(f'{"max_depth":<12} {"Acc. Train":<14} {"Acc. Test":<12} {"Observación"}')
print('-' * 60)

for depth in profundidades:
    dt = DecisionTreeClassifier(max_depth=depth, random_state=42)
    dt.fit(X_train_c, y_train_c)   # Árboles no necesitan normalización
    acc_train = accuracy_score(y_train_c, dt.predict(X_train_c))
    acc_test  = accuracy_score(y_test_c,  dt.predict(X_test_c))
    gap = acc_train - acc_test
    obs = '⚠️ Overfitting' if gap > 0.05 else '✅ Bien'
    etiq = str(depth) if depth is not None else 'Sin límite'
    print(f'{etiq:<12} {acc_train:.3f}          {acc_test:.3f}        {obs}')

In [ ]:
# 🔧 PARÁMETRO: cambia la profundidad y observa cómo cambia el árbol
MAX_DEPTH = 4

dt_final = DecisionTreeClassifier(max_depth=MAX_DEPTH, random_state=42)
dt_final.fit(X_train_c, y_train_c)
y_pred_dt = dt_final.predict(X_test_c)

print(f'=== ÁRBOL DE DECISIÓN (max_depth={MAX_DEPTH}) ===')
print(classification_report(y_test_c, y_pred_dt,
                             target_names=['No cancelada','Cancelada']))
print('✅ Árbol de Decisión entrenado')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# --- Visualización del árbol ---
plot_tree(dt_final,
          feature_names=X_clas.columns.tolist(),
          class_names=['No cancel.', 'Cancelada'],
          filled=True, rounded=True, fontsize=7,
          ax=axes[0])
axes[0].set_title(f'Árbol de Decisión (max_depth={MAX_DEPTH})',
                  fontsize=12, fontweight='bold')

# --- Matriz de confusión ---
cm_dt = confusion_matrix(y_test_c, y_pred_dt)
sns.heatmap(cm_dt, annot=True, fmt='d', cmap='Greens', ax=axes[1],
            xticklabels=['Pred: No cancel.','Pred: Cancelada'],
            yticklabels=['Real: No cancel.','Real: Cancelada'])
axes[1].set_title('Matriz de Confusión\nÁrbol de Decisión', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Valor real')
axes[1].set_xlabel('Predicción')

plt.tight_layout()
plt.show()

### 💬 Reflexión 3
> 1. **Mira el árbol visualizado. ¿Cuál es la primera pregunta que hace el modelo?** ¿Tiene sentido de negocio?
> 2. **¿Por qué el accuracy en train sube cuando aumentas `max_depth` pero el de test no sube igual?**
>
> *(Escribe tu respuesta aquí)*

---

---
## PARTE 5 · Random Forest: clasificación robusta 🟠
**⏱ 8 minutos**

Un **Random Forest** entrena muchos árboles de decisión en paralelo sobre subconjuntos aleatorios de los datos y combina sus predicciones por votación mayoritaria. Así reduce el overfitting que sufre un árbol individual.

```
Árbol 1 → vota: Cancelada
Árbol 2 → vota: No cancelada     →  votación → Cancelada (2 de 3)
Árbol 3 → vota: Cancelada
```

**Parámetro clave:** `n_estimators` = número de árboles. Más árboles = más estable, pero más lento.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# 🔧 PARÁMETRO: prueba con 50, 100, 200 y observa si cambian las métricas
N_ESTIMATORS = 100

rf = RandomForestClassifier(
    n_estimators=N_ESTIMATORS,   # Número de árboles en el bosque
    random_state=42,             # Reproducibilidad
    n_jobs=-1                    # Usa todos los núcleos disponibles
)
rf.fit(X_train_c, y_train_c)    # Sin normalización — árboles son invariantes a la escala
y_pred_rf = rf.predict(X_test_c)

print(f'=== RANDOM FOREST ({N_ESTIMATORS} árboles) ===')
print(classification_report(y_test_c, y_pred_rf,
                             target_names=['No cancelada','Cancelada']))
print('✅ Random Forest entrenado')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# --- Importancia de features ---
importancias = pd.Series(rf.feature_importances_, index=X_clas.columns)
top20 = importancias.nlargest(20).sort_values()

top20.plot(kind='barh', color='#E65100', edgecolor='white', ax=axes[0])
axes[0].set_title('Top 20 Features más importantes\n(Random Forest)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Importancia (Gini)')
axes[0].grid(True, alpha=0.3, axis='x')

# --- Matriz de confusión ---
cm_rf = confusion_matrix(y_test_c, y_pred_rf)
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Oranges', ax=axes[1],
            xticklabels=['Pred: No cancel.','Pred: Cancelada'],
            yticklabels=['Real: No cancel.','Real: Cancelada'])
axes[1].set_title('Matriz de Confusión\nRandom Forest', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Valor real')
axes[1].set_xlabel('Predicción')

plt.tight_layout()
plt.show()

---
## PARTE 6 · Comparación de clasificadores: ROC y tabla resumen 📊
**⏱ 8 minutos**

La **curva ROC** muestra el tradeoff entre la tasa de verdaderos positivos (sensibilidad) y la tasa de falsos positivos para distintos umbrales de decisión. El **AUC** (área bajo la curva) resume en un solo número qué tan bien separa el modelo las dos clases:
- AUC = 1.0 → modelo perfecto
- AUC = 0.5 → modelo aleatorio
- AUC < 0.5 → peor que el azar

In [ ]:
from sklearn.metrics import roc_curve, auc, accuracy_score, f1_score, precision_score, recall_score

# Probabilidades predichas (para la curva ROC necesitamos prob, no clase)
prob_lr = lr.predict_proba(X_test_c_sc)[:, 1]   # Regresión Logística (usa datos escalados)
prob_dt = dt_final.predict_proba(X_test_c)[:, 1] # Árbol (sin escalar)
prob_rf = rf.predict_proba(X_test_c)[:, 1]       # Random Forest (sin escalar)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- Curvas ROC ---
modelos_roc = [
    ('Reg. Logística', prob_lr, '#1565C0'),
    ('Árbol Decisión', prob_dt, '#2E7D32'),
    ('Random Forest',  prob_rf, '#E65100'),
]

for nombre, probs, color in modelos_roc:
    fpr, tpr, _ = roc_curve(y_test_c, probs)
    auc_score   = auc(fpr, tpr)
    axes[0].plot(fpr, tpr, color=color, linewidth=2,
                 label=f'{nombre} (AUC={auc_score:.3f})')

# Línea base (clasificador aleatorio)
axes[0].plot([0,1],[0,1],'--', color='gray', linewidth=1, label='Aleatorio (AUC=0.5)')
axes[0].set_xlabel('Tasa de Falsos Positivos', fontsize=11)
axes[0].set_ylabel('Tasa de Verdaderos Positivos', fontsize=11)
axes[0].set_title('Curvas ROC — Comparación de clasificadores', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# --- Tabla de métricas ---
preds = [
    ('Reg. Logística', y_pred_lr),
    ('Árbol Decisión', y_pred_dt),
    ('Random Forest',  y_pred_rf),
]

nombres, accs, precs, recs, f1s, aucs = [], [], [], [], [], []
for nombre, pred in preds:
    nombres.append(nombre)
    accs.append(f'{accuracy_score(y_test_c, pred):.3f}')
    precs.append(f'{precision_score(y_test_c, pred):.3f}')
    recs.append(f'{recall_score(y_test_c, pred):.3f}')
    f1s.append(f'{f1_score(y_test_c, pred):.3f}')

for (nombre, probs, _), p in zip(modelos_roc, [prob_lr, prob_dt, prob_rf]):
    fpr, tpr, _ = roc_curve(y_test_c, probs)
    aucs.append(f'{auc(fpr, tpr):.3f}')

tabla = pd.DataFrame({
    'Modelo': nombres, 'Accuracy': accs,
    'Precision': precs, 'Recall': recs, 'F1': f1s, 'AUC': aucs
})

# Mostrar tabla como imagen en el segundo subplot
axes[1].axis('off')
t = axes[1].table(cellText=tabla.values, colLabels=tabla.columns,
                  loc='center', cellLoc='center')
t.auto_set_font_size(False)
t.set_fontsize(10)
t.scale(1.2, 2.0)
axes[1].set_title('Tabla de métricas — Clasificadores', fontsize=12, fontweight='bold', pad=20)

plt.tight_layout()
plt.show()

print('\n=== TABLA DETALLADA ===')
print(tabla.to_string(index=False))

### 💬 Reflexión 4
> 1. **¿Cuál clasificador tiene mejor AUC? ¿Y mejor F1?** ¿Son siempre el mismo?
> 2. **Un hotel prefiere no perder ingresos por cancelaciones imprevistas. ¿Le interesa maximizar el Recall o la Precision?** ¿Por qué?
>
> *(Escribe tu respuesta aquí)*

---

---
## PARTE 7 · Regresión: predecir el precio por noche 🔴
**⏱ 10 minutos**

Cambiamos de problema: ahora queremos predecir un **valor numérico continuo** (precio por noche) en lugar de una clase. Las métricas de evaluación también cambian:

| Métrica | Qué mide | Unidad |
|---|---|---|
| MAE | Error absoluto medio | Euros |
| RMSE | Raíz del error cuadrático medio (penaliza errores grandes) | Euros |
| R² | Proporción de varianza explicada (1.0 = perfecto, 0 = nada) | Sin unidades |

Usaremos tres algoritmos: **Regresión Lineal**, **Árbol de Regresión** y **Random Forest Regresivo**.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# === REGRESIÓN LINEAL ===
rl = LinearRegression()
rl.fit(X_train_r_sc, y_train_r)      # Usa datos normalizados
y_pred_rl = rl.predict(X_test_r_sc)

# === ÁRBOL DE REGRESIÓN ===
# 🔧 PARÁMETRO: ajusta max_depth para ver overfitting/underfitting
MAX_DEPTH_REG = 5
dtr = DecisionTreeRegressor(max_depth=MAX_DEPTH_REG, random_state=42)
dtr.fit(X_train_r, y_train_r)        # Sin normalizar
y_pred_dtr = dtr.predict(X_test_r)

# === RANDOM FOREST REGRESIVO ===
rfr = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rfr.fit(X_train_r, y_train_r)        # Sin normalizar
y_pred_rfr = rfr.predict(X_test_r)

def metricas_regresion(nombre, y_real, y_pred):
    mae  = mean_absolute_error(y_real, y_pred)
    rmse = mean_squared_error(y_real, y_pred) ** 0.5
    r2   = r2_score(y_real, y_pred)
    print(f'{nombre:<25} MAE: €{mae:6.2f}  RMSE: €{rmse:6.2f}  R²: {r2:.3f}')
    return mae, rmse, r2

print('=== COMPARACIÓN DE MODELOS DE REGRESIÓN ===')
m_rl  = metricas_regresion('Regresión Lineal',        y_test_r, y_pred_rl)
m_dtr = metricas_regresion(f'Árbol (depth={MAX_DEPTH_REG})', y_test_r, y_pred_dtr)
m_rfr = metricas_regresion('Random Forest',           y_test_r, y_pred_rfr)
print('\n✅ Tres modelos de regresión entrenados')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Gráfico pred vs real para cada modelo
modelos_reg = [
    ('Regresión Lineal', y_pred_rl,  '#1565C0', m_rl),
    (f'Árbol (depth={MAX_DEPTH_REG})', y_pred_dtr, '#2E7D32', m_dtr),
    ('Random Forest',    y_pred_rfr, '#E65100', m_rfr),
]

y_min = y_test_r.min()
y_max = y_test_r.max()

for ax, (nombre, pred, color, m) in zip(axes, modelos_reg):
    mae, rmse, r2 = m
    ax.scatter(y_test_r, pred, alpha=0.5, s=40, c=color, edgecolors='white', linewidth=0.3)
    # Línea perfecta (pred = real)
    ax.plot([y_min, y_max], [y_min, y_max], '--', color='black', linewidth=1.2, label='Predicción perfecta')
    ax.set_xlabel('Precio real (€/noche)', fontsize=10)
    ax.set_ylabel('Precio predicho (€/noche)', fontsize=10)
    ax.set_title(f'{nombre}\nMAE=€{mae:.1f}  RMSE=€{rmse:.1f}  R²={r2:.3f}',
                 fontsize=11, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Predicción de precio por noche: Real vs Predicho', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Importancia de features en Random Forest Regresivo
imp_reg = pd.Series(rfr.feature_importances_, index=X_reg.columns).nlargest(15).sort_values()

fig, ax = plt.subplots(figsize=(10, 5))
imp_reg.plot(kind='barh', color='#BF360C', edgecolor='white', ax=ax)
ax.set_title('Top 15 Features para predecir precio/noche\n(Random Forest Regresivo)',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Importancia')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

### 💬 Reflexión 5
> 1. **¿Por qué el R² de la Regresión Lineal es bajo si el dataset tiene precio_noche como columna continua?**
>    Pista: revisa cómo se generó `precio_noche` — ¿tiene alguna relación lineal con las otras features?
> 2. **¿Cuáles son las features más importantes para predecir el precio? ¿Tiene sentido de negocio?**
>
> *(Escribe tu respuesta aquí)*

---

---
## PARTE 8 · Comparación final: supervisado vs no supervisado 🏆
**⏱ 5 minutos**

Miramos en perspectiva todo lo que vimos en las dos actividades.

In [ ]:
print('=' * 70)
print('   COMPARACIÓN GENERAL: SUPERVISADO vs NO SUPERVISADO')
print('=' * 70)

tabla_final = pd.DataFrame({
    'Característica': [
        '¿Necesita etiquetas?',
        '¿Qué aprende?',
        'Ejemplo en hotelería',
        'Evaluación',
        'Algoritmos (esta actividad)'
    ],
    'Supervisado': [
        'Sí (cancelado, precio)',
        'A mapear features → target',
        'Predecir si habrá cancelación',
        'Accuracy, F1, AUC, R², RMSE',
        'Log. Reg. / Árbol / Random Forest'
    ],
    'No supervisado': [
        'No',
        'Estructura oculta en los datos',
        'Agrupar huéspedes por perfil',
        'Silhouette, inercia, dendrograma',
        'K-Means / Jerárquico / DBSCAN'
    ]
})
print(tabla_final.to_string(index=False))

print('\n=== RESUMEN DE MODELOS DE CLASIFICACIÓN (test set) ===')
print(tabla.to_string(index=False))

print('\n=== RESUMEN DE MODELOS DE REGRESIÓN (test set) ===')
print(f'{"Modelo":<25} {"MAE":>8} {"RMSE":>8} {"R²":>8}')
print('-' * 55)
nombres_reg = ['Regresión Lineal', f'Árbol (depth={MAX_DEPTH_REG})', 'Random Forest']
for nombre, m in zip(nombres_reg, [m_rl, m_dtr, m_rfr]):
    print(f'{nombre:<25} €{m[0]:>6.2f}  €{m[1]:>6.2f}  {m[2]:>7.3f}')

---
## 💬 Reflexión Final

> **1. El Random Forest suele ganar en métricas, pero un árbol simple es más interpretable. ¿Cuándo elegirías uno sobre el otro en un contexto real de hotel?**
>
> *(Escribe aquí)*

> **2. Imagina que el director del hotel dice: "prefiero equivocarme diciéndole a un huésped que se va a cancelar cuando no era así, antes que no detectar una cancelación real". ¿Qué métrica priorizarías y cómo ajustarías el umbral de decisión?**
>
> *(Escribe aquí)*

> **3. ¿Cuál es la diferencia fundamental entre lo que aprendiste en la actividad de ML no supervisado y esta? Usa un ejemplo concreto del dataset de hotel.**
>
> *(Escribe aquí)*

> **4. ¿Qué pasaría si intentaras predecir `cancelado` sin separar train y test? ¿Por qué eso sería engañoso?**
>
> *(Escribe aquí)*

---

## ✅ ¡Actividad completada!

| | Lo que hiciste hoy |
|---|---|
| 🔵 Regresión Logística | Clasificaste cancelaciones con un modelo lineal interpretable |
| 🟢 Árbol de Decisión | Construiste reglas de negocio automáticas y viste el overfitting |
| 🟠 Random Forest | Combinaste 100 árboles para una predicción más robusta |
| 📊 Curvas ROC | Comparaste clasificadores más allá del accuracy simple |
| 🔴 Regresión | Predijiste precios con tres enfoques distintos y evaluaste con MAE, RMSE, R² |

---
## 🎯 RETOS OPCIONALES

Elige uno y experimenta en la celda de abajo:

**A. Ajuste de umbral de decisión**
Por defecto los clasificadores usan 0.5 como umbral. Cambia el umbral a 0.3 y observa cómo suben el Recall y bajan la Precision:
```python
umbral = 0.3  # 🔧 prueba 0.2, 0.4, 0.6
y_pred_umbral = (rf.predict_proba(X_test_c)[:, 1] >= umbral).astype(int)
print(classification_report(y_test_c, y_pred_umbral))
```

**B. Cross-validation**
Evalúa el Random Forest con 5-fold cross-validation para una estimación más confiable del rendimiento:
```python
from sklearn.model_selection import cross_val_score
scores = cross_val_score(rf, X_clas, y_clas, cv=5, scoring='f1')
print(f'F1 medio: {scores.mean():.3f} ± {scores.std():.3f}')
```

**C. GridSearchCV en el Árbol**
Encuentra automáticamente el mejor `max_depth` entre 2 y 10:
```python
from sklearn.model_selection import GridSearchCV
param_grid = {'max_depth': range(2, 11)}
grid = GridSearchCV(DecisionTreeClassifier(random_state=42), param_grid, cv=5, scoring='f1')
grid.fit(X_train_c, y_train_c)
print(f'Mejor profundidad: {grid.best_params_}  F1: {grid.best_score_:.3f}')
```

In [ ]:
# 🔧 Espacio para tus experimentos
